In [1]:
import sys
import os




project_path = r"C:\Users\lamin\OneDrive\Documents\Maitrise en info Canada\ASTD project\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb

YEAR = 2019

MONTHS_TO_LOAD = [5, 6]

USECOLS   = "default"
SAMPLING  = [0, -1]
QUAL_MIN  = 60

BASE_PATH = r"C:\Users\lamin\OneDrive\Documents\Maitrise en info Canada\ASTD project\data"
df = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=SAMPLING, quality_threshold_minutes=QUAL_MIN
)

# Build tracks and collect processing logs
print("Building ship tracks (this may take a while)...")
tracks, logs = tb.build_ship_tracks(
    df,
    max_time_gap_hours=48,
    max_distance_km=600,
    min_track_length=2,
    matching_strategy="balanced",
    return_logs=True,
)





Loading ASTD CSVs:   0%|          | 0/2 [00:00<?, ?it/s]

Building ship tracks (this may take a while)...
Data sample after cleaning:
  Date range: 2019-05-01 00:00:00+00:00 to 2019-06-30 23:59:57+00:00
  Ship types: ['unknown' 'fishing vessels' 'bulk carriers' 'general cargo ships'
 'other activities' 'chemical tankers' 'crude oil tankers'
 'passenger ships' 'ro-ro cargo ships' 'other service offshore vessels'
 'offshore supply ships' 'oil product tankers' 'gas tankers'
 'container ships' 'refrigerated cargo ships' 'cruise ships']
  Unique ships: 3888
Creating segments for 3888 unique shipids
Created 3888 segments
Sample segment: unknown|nan|nan|nan


C:\Users\lamin\OneDrive\Documents\Maitrise en info Canada\ASTD project\TrackBuilder\track_builder\core\track_helpers.py:107: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')


In [2]:
df.head()

,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
0,618,2019-05-01 00:00:00+00:00,NaN,NaN,Unknown,NaN,3.007342,361,7.729249,63.120972
1,1877,2019-05-01 00:00:01+00:00,Iceland,NaN,Fishing vessels,< 1000 GT,1119.830688,310,-22.486521,64.416367
2,47,2019-05-01 00:00:01+00:00,NaN,NaN,Unknown,NaN,3.148481,611,24.705662,60.166294
3,3579,2019-05-01 00:00:01+00:00,Norway,NaN,Fishing vessels,1000 - 4999 GT,2.606242,541,12.558700,66.166725
4,321,2019-05-01 00:00:02+00:00,NaN,NaN,Unknown,NaN,0.478249,121,-18.909559,66.148773


In [3]:

typical_speeds = tb.track._compute_typical_speeds_from_data(df)

C:\Users\lamin\OneDrive\Documents\Maitrise en info Canada\ASTD project\TrackBuilder\track_builder\core\track_helpers.py:107: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')


In [6]:
print("Typical speeds (km/h):")
typical_speeds

Typical speeds (km/h):


{'unknown': 5.055167667116676,
 'fishing vessels': 9.184696871628129,
 'offshore supply ships': 12.076239279061884,
 'passenger ships': 12.617244556863428,
 'other activities': 15.614746079905094,
 'chemical tankers': 17.09406537227542,
 'other service offshore vessels': 17.787406328805226,
 'crude oil tankers': 18.407327891659136,
 'refrigerated cargo ships': 19.748455011442044,
 'gas tankers': 21.064923706228182,
 'cruise ships': 22.01777378627486,
 'general cargo ships': 24.70287199859417,
 'bulk carriers': 26.64914716759849,
 'oil product tankers': 27.907406132433746,
 'ro-ro cargo ships': 29.85179766593972,
 'container ships': 30.266038888679013}

In [ ]:

import plotly.graph_objects as go
import pandas as pd
import numpy as np
import track_builder as tb
from track_builder.core.track_helpers import _compute_speed_kmh_between_rows, to_ts


# Make sure 'df' is your main astd_data DataFrame,
# loaded with all necessary columns (e.g., from config.ASTD_USEFUL_COLS)

# 1. Get the raw speed data
# n_per_day=20: We sample 20 speed points per ship per day.
# This is to make plotting feasible.
print("Computing all point-to-point speeds (this may take a minute)...")
speeds_df = tb.core.track_helpers.get_all_point_to_point_speeds(df, n_per_day=20) 
print(f"Calculation complete. Found {len(speeds_df)} valid speed points.")

# 2. Create the box plot
fig = go.Figure()

# Sort categories by their median speed for a cleaner plot
if not speeds_df.empty:
    # Ensure 'astd_cat' is treated as a string for grouping
    speeds_df['astd_cat'] = speeds_df['astd_cat'].astype(str)
    categories = speeds_df.groupby('astd_cat')['speed_kmh'].median().sort_values().index
else:
    categories = []
    print("Warning: No speed data found to plot.")

for cat in categories:
    fig.add_trace(go.Box(
        y=speeds_df[speeds_df['astd_cat'] == cat]['speed_kmh'],
        name=cat,
        boxpoints='outliers', # This is the key: show *only* outlier points
        jitter=0.3,
        pointpos=-1.8,
        marker=dict(size=4)
    ))

fig.update_layout(
    title="Distribution of Point-to-Point Speeds by Category (with Outliers)",
    yaxis_title="Speed (km/h) (Filtered > 0 and <= 110)",
    xaxis_title="Ship Category (ASTD)",
    height=600,
    showlegend=False,
    # Order the x-axis by the median speed we calculated
    xaxis=dict(
        categoryorder='array', 
        categoryarray=categories
    ) 
)

fig.show()

Computing all point-to-point speeds (this may take a minute)...


[track_helpers] Computing p-t-p speeds: 100%|██████████| 4815/4815 [00:04<00:00, 988.48it/s] 


Calculation complete. Found 90754 valid speed points.


In [8]:
logs

,match_id,from_shipid,to_shipid,from_month,to_month,stage,reason,dt_hours,distance_km_fd,implied_v_kmh
0,618→603,618.0,603.0,2019-05,2019-06,filter,time_window,720.011667,261.379150,NaN
1,618→1045,618.0,1045.0,2019-05,2019-06,filter,time_window,720.011944,683.933899,NaN
2,618→1064,618.0,1064.0,2019-05,2019-06,filter,time_window,720.013056,746.997803,NaN
3,618→600,618.0,600.0,2019-05,2019-06,filter,time_window,720.013056,63.919846,NaN
4,618→1739,618.0,1739.0,2019-05,2019-06,filter,time_window,720.013611,1295.384399,NaN
...,...,...,...,...,...,...,...,...,...,...
3699011,3540→3913,NaN,NaN,NaN,NaN,skip,already_assigned,NaN,NaN,NaN
3699012,3540→632,NaN,NaN,NaN,NaN,skip,already_assigned,NaN,NaN,NaN
3699013,3540→837,NaN,NaN,NaN,NaN,skip,already_assigned,NaN,NaN,NaN
3699014,3540→641,NaN,NaN,NaN,NaN,skip,already_assigned,NaN,NaN,NaN


In [9]:
stats = tb.get_track_statistics(tracks, df)
print(stats["n_tracks"], stats["avg_length"], stats["max_length"])

623 2.0 2


In [10]:
tracks.head()

,month,segment_id,track_id
0,2019-05,1877,2
1,2019-06,2070,2
2,2019-05,1885,10
3,2019-06,1933,10
4,2019-05,1837,11


In [12]:
# Candidates for a segment
cands, clog = tb.find_track_candidates(1877, "2019-05", df, top_n=5, return_logs=True)

Data sample after cleaning:
  Date range: 2019-05-01 00:00:00+00:00 to 2019-06-30 23:59:57+00:00
  Ship types: ['unknown' 'fishing vessels' 'bulk carriers' 'general cargo ships'
 'other activities' 'chemical tankers' 'crude oil tankers'
 'passenger ships' 'ro-ro cargo ships' 'other service offshore vessels'
 'offshore supply ships' 'oil product tankers' 'gas tankers'
 'container ships' 'refrigerated cargo ships' 'cruise ships']
  Unique ships: 3888
Creating segments for 3888 unique shipids
Created 3888 segments
Sample segment: unknown|nan|nan|nan


C:\Users\lamin\OneDrive\Documents\Maitrise en info Canada\ASTD project\TrackBuilder\track_builder\core\track_helpers.py:107: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [13]:
cands

,month,segment_id,match_score_simple,match_score_core,distance_km_fd,implied_v_kmh,dt_hours
0,2019-06,2070,0.010407,NaN,0.060353,0.413847,0.145833
1,2019-06,1883,0.010440,NaN,0.042561,0.436520,0.097500
2,2019-06,3957,0.018142,NaN,0.079413,0.787565,0.100833
3,2019-06,1893,0.066528,NaN,0.313346,3.000125,0.104444
4,2019-06,1880,0.088381,NaN,0.379419,4.005596,0.094722


In [14]:
clog

,match_id,from_shipid,to_shipid,from_month,to_month,stage,reason,dt_hours,distance_km_fd,implied_v_kmh
0,1877→603,1877,603,2019-05,2019-06,filter,distance_window,0.091111,1600.061890,NaN
1,1877→3795,1877,3795,2019-05,2019-06,filter,distance_window,0.091389,2241.420898,NaN
2,1877→1045,1877,1045,2019-05,2019-06,filter,distance_window,0.091389,2070.852783,NaN
3,1877→1046,1877,1046,2019-05,2019-06,filter,distance_window,0.091944,2009.173462,NaN
4,1877→4556,1877,4556,2019-05,2019-06,filter,distance_window,0.092222,1453.748901,NaN
...,...,...,...,...,...,...,...,...,...,...
788,1877→2639,1877,2639,2019-05,2019-06,filter,speed_cap,0.191667,290.696259,1516.676132
789,1877→226,1877,226,2019-05,2019-06,filter,speed_cap,0.191944,304.599060,1586.912614
790,1877→3393,1877,3393,2019-05,2019-06,filter,speed_cap,0.205556,212.586212,1034.203194
791,1877→4448,1877,4448,2019-05,2019-06,filter,speed_cap,0.963056,529.053711,549.349109


In [15]:
# segments for track_id 2
t2 = tracks[tracks['track_id']==2].sort_values('month')
print(t2)

# look at decisions around 1885 (May) -> 1933 (June)
mask = (logs['from_shipid'] == float(1877)) | (logs['to_shipid'] == float(1883))
print(logs[mask].head(20))


     month  segment_id  track_id
0  2019-05        1877         2
1  2019-06        2070         2
        match_id  from_shipid  to_shipid from_month to_month   stage  \
412     618→1883        618.0     1883.0    2019-05  2019-06  filter   
1679   1877→5628       1877.0     5628.0    2019-05  2019-06  filter   
1680  1877→17343       1877.0    17343.0    2019-05  2019-06  filter   
1681  1877→13931       1877.0    13931.0    2019-05  2019-06  filter   
1682  1877→13442       1877.0    13442.0    2019-05  2019-06  filter   
1683   1877→8970       1877.0     8970.0    2019-05  2019-06  filter   
1684  1877→12866       1877.0    12866.0    2019-05  2019-06  filter   
1685  1877→18618       1877.0    18618.0    2019-05  2019-06  filter   
1686  1877→12939       1877.0    12939.0    2019-05  2019-06  filter   
1687  1877→14042       1877.0    14042.0    2019-05  2019-06  filter   
1688   1877→7951       1877.0     7951.0    2019-05  2019-06  filter   
1689  1877→11005       1877.0    1100

In [16]:
# choose a track id to visualize
fig_track = tb.plot_individual_track(
    2,
    tracks,
    df,
    show_segments=True,                 # one polyline per segment (often = shipid)
    map_style="open-street-map",
    title=f"Track {2} – vue détaillée",
)


fig_track.show()


In [17]:
import pandas as pd
# visualize tracks for two ships only

# choose id = 1877 (May 2019) which was well tracked and only the last day of May
df_1877_filtered = df[(df['shipid'] == 1877) & (df['date_time_utc'].dt.month == 5) & (df['date_time_utc'].dt.day == 31)]

# choose id = 2070 (June 2019) which was well tracked and only the first day of June
df_2070_filtered = df[(df['shipid'] == 2070) & (df['date_time_utc'].dt.month == 6) & (df['date_time_utc'].dt.day == 1)]

#  combine the two filtered dataframes
df_filtered = pd.concat([df_1877_filtered, df_2070_filtered])

fig_track2 = tb.plot_ship_tracks(
    df_filtered,
    show_points=True,
    color_by="shipid",
    title="Trajectoires des navires 1877 et 2070",
    map_style="open-street-map",
)

fig_track2.show()


In [18]:
# visualize tracks for two ships only
df_1883_filtered = df[(df['shipid'] == 1883) & (df['date_time_utc'].dt.month == 6) & (df['date_time_utc'].dt.day == 1)]

df_filtered2 = pd.concat([df_1877_filtered, df_1883_filtered])

fig_track3 = tb.plot_ship_tracks(
    df_filtered2,
    show_points=True,
    color_by="shipid",
    title="Trajectoires des navires 1877 et 2070",
    map_style="open-street-map",
)

fig_track3.show()


In [19]:
# visualize all points for ship 1877 in May 2019
df_1877 = df[(df['shipid'] == 1877) & (df['date_time_utc'].dt.month == 5)]

fig_1877 = tb.plot_ship_tracks(
    df_1877,
    show_points=True,
    color_by="shipid",
    map_style="open-street-map",
    date_from="2019-05-31",
)

df_1877.head(1)




,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
1,1877,2019-05-01 00:00:01+00:00,Iceland,NaN,Fishing vessels,< 1000 GT,1119.830688,310,-22.486521,64.416367


In [20]:
fig_1877.show()

In [21]:
# choisir le navire avec id = 2070 (juin 2019) qui a été bien suivi
df_2070 = df[(df['shipid'] == 2070) & (df['date_time_utc'].dt.month == 6)]

fig_2070 = tb.plot_ship_tracks(
    df_2070,
    show_points=True,
    color_by="shipid",
    map_style="open-street-map",
    date_to="2019-06-01 23:59:59", #added time to include full day
)

df_2070.head(1)




,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
475332,2070,2019-06-01 00:03:18+00:00,Iceland,NaN,Fishing vessels,< 1000 GT,1.542593,121,-22.420359,63.839466


In [ ]:
fig_2070.show()

In [22]:
# choisir le navire avec id = 2070 (juin 2019) qui a été bien suivi
df_1883 = df[(df['shipid'] == 1883) & (df['date_time_utc'].dt.month == 6)]

fig_1883 = tb.plot_ship_tracks(
    df_1883,
    show_points=True,
    color_by="shipid",
    map_style="open-street-map",
    date_to="2019-06-01 23:59:59", # added time to include full day
)

df_1883.head(1)




,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
474943,1883,2019-06-01 00:00:24+00:00,Iceland,NaN,Fishing vessels,< 1000 GT,0.380528,39,-22.419182,63.839184


In [ ]:
fig_1883.show()

# =========== load 5 month =============